Unfortunately I did not select this submission: it would have placed me top 22 with a score of 69.2 the CV score being 61.9 in 5 fold

# Almost only trends


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import amp_pd_peptide

from scipy.optimize import minimize


In [ ]:
def smape_plus_1(y_true, y_pred):
    y_true_plus_1 = y_true + 1
    y_pred_plus_1 = y_pred + 1
    metric = np.zeros(len(y_true_plus_1))
    
    numerator = np.abs(y_true_plus_1 - y_pred_plus_1)
    denominator = ((np.abs(y_true_plus_1) + np.abs(y_pred_plus_1)) / 2)
    
    mask_not_zeros = (y_true_plus_1 != 0) | (y_pred_plus_1 != 0)
    metric[mask_not_zeros] = numerator[mask_not_zeros] / denominator[mask_not_zeros]
    
    return 100 * np.nanmean(metric)

In [ ]:
train_clinical_data = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/train_clinical_data.csv')
train_clinical_data['source'] = 'standard'

supplemental_clinical_data = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/supplemental_clinical_data.csv')
supplemental_clinical_data['source'] = 'supplemental'

train_clinical_all = pd.concat([train_clinical_data, supplemental_clinical_data])

In [ ]:
# delete visit_month 3, 5, 9 (there are no such visit_months in the Test API)
train_clinical_all = train_clinical_all[~train_clinical_all.visit_month.isin([3, 5, 9])]

In [ ]:
train_clinical_all['pred_month'] = train_clinical_all['visit_month']

for plus_month in [6, 12, 24]:
    train_shift = train_clinical_all[['patient_id', 'visit_month', 'pred_month', 'updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']].copy()
    train_shift['visit_month'] -= plus_month
    train_shift.rename(columns={f'updrs_{i}': f'updrs_{i}_plus_{plus_month}' for i in range(1, 5)}, inplace=True)
    train_shift.rename(columns={'pred_month': f'pred_month_plus_{plus_month}'}, inplace=True)
    train_clinical_all = train_clinical_all.merge(train_shift, how='left', on=['patient_id', 'visit_month'])

train_clinical_all.rename(columns={f'updrs_{i}': f'updrs_{i}_plus_0' for i in range(1, 5)}, inplace=True)
train_clinical_all.rename(columns={'pred_month': f'pred_month_plus_0'}, inplace=True)
train_clinical_all

# Stage1
Trend


In [ ]:
def calculate_month_trend_predicitons(pred_month, trend, target):
    if target == 'updrs_1': pred_month = pred_month.clip(12, 36)
    if target == 'updrs_2': pred_month = pred_month.clip(40, None)
    if target == 'updrs_3': pred_month = pred_month.clip(1, None)
    if target == 'updrs_4': pred_month = pred_month.clip(55, None)
    #pred_month = pred_month.clip(None, c_m)
    return np.round(trend[0] + pred_month * trend[1] \
                    + pred_month**2 * trend[2] \
                    + np.sin(pred_month*trend[4]+trend[5]) * trend[3] \
                    + np.exp(-pred_month*trend[6]+trend[7]) * trend[8] \
                   ) #+ np.cos(pred_month*trend[4] +trend[5]) * trend[3])

def function_to_minimize(x):    
    metric = smape_plus_1(
        y_true=y_true_array, 
        y_pred=calculate_month_trend_predicitons(
            pred_month=pred_month_array,
            trend=x,
            target=target,
        )
    )
    return metric

off_pred = list()
y_to_pred = list()
preds = np.zeros([len(train_clinical_all),16])
pred_col = list()
target_to_trend = {}

for i in range(1, 5):
    if True:
    
        target = f'updrs_{i}'
        columns_with_target = [f'{target}_plus_{plus_month}' for plus_month in [0, 6, 12, 24]]
        columns_with_pred_month = [f'pred_month_plus_{plus_month}' for plus_month in [0, 6, 12, 24]]
        
        y_true_array = train_clinical_all[columns_with_target].values.ravel()
        pred_month_array = train_clinical_all[columns_with_pred_month].values.ravel()
        optimized = minimize(fun=function_to_minimize,
                             x0=[0, 0, 0, 0, 0 , 0, 0 ,0,0],
                             method='Powell',
                            )
        for y in y_true_array:
            y_to_pred.append(y)
        pred = calculate_month_trend_predicitons(pred_month=pred_month_array,trend=optimized.x,target=target)
        for y in pred:
            off_pred.append(y)

        preds[:,(i-1)*4:i*4] = pred.reshape(-1,4)
        trend = list(optimized.x)
        target_to_trend[target] = trend
        print("SMAPE+1 for updrs_{} = {}".format(i,optimized.fun))

print("OOF SMAPE+1 : {}".format(smape_plus_1(y_true=np.array(y_to_pred),
                                             y_pred=np.array(off_pred))))
    
#target_to_trend

    OOF SMAPE+1 : 55.9651099370812 linear + clip
    OOF SMAPE+1 : 55.86833325869621 + ^2 sin exp(-
    OOF SMAPE+1 : 55.84713160568972 + more clip

In [ ]:
columns_with_target = list()
for i in range(1,5):
    target = f'updrs_{i}'
    pred_col += [f'{target}_plus_{plus_month}_s1' for plus_month in [0, 6, 12, 24]]

train_clinical_all[pred_col] = preds

# eval oof pred
true_col = [f'{target}_plus_{plus_month}' for plus_month in [0, 6, 12, 24] for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']]
y_gt = train_clinical_all[true_col].values.ravel()
pred_col = [f'{target}_plus_{plus_month}'+"_s1" for plus_month in [0, 6, 12, 24] for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']]
preds = train_clinical_all[pred_col].values.ravel()
print("Stage1 oof pred : {}".format(smape_plus_1(y_gt,preds)))

# Satge2

Protein NPX XGB

In [ ]:
len(train_clinical_all)

In [ ]:
proteins = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/train_proteins.csv')
proteins_features = pd.pivot_table(proteins, values='NPX', index='visit_id', columns='UniProt', aggfunc='sum')

train_clinical_all = train_clinical_all.merge(
    proteins_features,
    left_on='visit_id',
    right_index=True,
    how='inner' # ?
)

# fill with last valid :
# we can do better with the count of how many mounts have past for a model
train_clinical_all[proteins_features.columns] = train_clinical_all.groupby('patient_id')[proteins_features.columns].fillna(method='ffill')
train_clinical_all = train_clinical_all.fillna(-1)

In [ ]:
len(train_clinical_all)

In [ ]:
uni_prot = np.array(['O00391', 'O00533', 'O00584', 'O14498', 'O14773', 'O14791',
       'O15240', 'O15394', 'O43505', 'O60888', 'O75144', 'O75326',
       'O94919', 'P00441', 'P00450', 'P00734', 'P00736', 'P00738',
       'P00746', 'P00747', 'P00748', 'P00751', 'P01008', 'P01009',
       'P01011', 'P01019', 'P01023', 'P01024', 'P01031', 'P01033',
       'P01034', 'P01042', 'P01344', 'P01591', 'P01608', 'P01621',
       'P01717', 'P01780', 'P01833', 'P01834', 'P01857', 'P01859',
       'P01860', 'P01861', 'P01876', 'P01877', 'P02452', 'P02647',
       'P02649', 'P02652', 'P02655', 'P02656', 'P02671', 'P02675',
       'P02679', 'P02747', 'P02748', 'P02749', 'P02750', 'P02751',
       'P02753', 'P02760', 'P02763', 'P02765', 'P02766', 'P02768',
       'P02774', 'P02787', 'P02790', 'P04004', 'P04075', 'P04156',
       'P04180', 'P04196', 'P04207', 'P04211', 'P04216', 'P04217',
       'P04275', 'P04406', 'P04433', 'P05060', 'P05067', 'P05090',
       'P05155', 'P05156', 'P05408', 'P05452', 'P05546', 'P06310',
       'P06396', 'P06454', 'P06681', 'P06727', 'P07195', 'P07225',
       'P07333', 'P07339', 'P07602', 'P07711', 'P07858', 'P07998',
       'P08123', 'P08133', 'P08253', 'P08294', 'P08493', 'P08571',
       'P08603', 'P08637', 'P08697', 'P09104', 'P09486', 'P09871',
       'P10451', 'P10643', 'P10645', 'P10909', 'P11142', 'P11277',
       'P12109', 'P13473', 'P13521', 'P13591', 'P13611', 'P13671',
       'P13987', 'P14174', 'P14314', 'P14618', 'P16035', 'P16070',
       'P16152', 'P16870', 'P17174', 'P17936', 'P18065', 'P19021',
       'P19652', 'P19823', 'P19827', 'P20774', 'P20933', 'P23083',
       'P23142', 'P24592', 'P25311', 'P27169', 'P30086', 'P31997',
       'P35542', 'P36222', 'P36955', 'P36980', 'P39060', 'P40925',
       'P41222', 'P43121', 'P43251', 'P43652', 'P49588', 'P49908',
       'P51884', 'P54289', 'P55290', 'P61278', 'P61626', 'P61769',
       'P61916', 'P80748', 'P98160', 'Q02818', 'Q06481', 'Q08380',
       'Q12805', 'Q12841', 'Q12907', 'Q13283', 'Q13332', 'Q13451',
       'Q13740', 'Q14118', 'Q14508', 'Q14515', 'Q14624', 'Q15904',
       'Q16270', 'Q16610', 'Q562R1', 'Q6UX71', 'Q6UXB8', 'Q6UXD5',
       'Q7Z3B1', 'Q7Z5P9', 'Q8IWV7', 'Q8N2S1', 'Q8NBJ4', 'Q8NE71',
       'Q92520', 'Q92823', 'Q92876', 'Q96BZ4', 'Q96KN2', 'Q96PD5',
       'Q96S96', 'Q99435', 'Q99674', 'Q99832', 'Q99969', 'Q9BY67',
       'Q9HDC9', 'Q9NQ79', 'Q9NYU2', 'Q9UBR2', 'Q9UBX5', 'Q9UHG2',
       'Q9UNU6', 'Q9Y646', 'Q9Y6R7', 'P01594', 'P02792', 'P32754',
       'P60174', 'Q13449', 'Q99683', 'Q99829', 'Q9UKV8'])

In [ ]:
stage1_f = pred_col
stage2_f = uni_prot.tolist() + stage1_f

train_s2 = train_clinical_all.copy()
train_clinical_all[stage2_f].head(2)

In [ ]:
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor

features = stage2_f
models = list()
estimators = [('lr', XGBRegressor()),
              ('svr', SVR())]

for i in range(1, 5):
    model = StackingRegressor(estimators=estimators,n_jobs=2)
    X = np.empty([len(train_s2)*4,len(features)+1])
    y = np.empty([len(train_s2)*4])
    target = f'updrs_{i}'
    for i,m in enumerate([0, 6, 12, 24]):
        X[len(train_s2)*i:len(train_s2)*(i+1),:-1] = train_s2[features].values
        X[len(train_s2)*i:len(train_s2)*(i+1),-1] = m
        y[len(train_s2)*i:len(train_s2)*(i+1)] = train_s2[f'{target}_plus_{m}'].values
    #emb = model.fit_transform(X,y)
    model.fit(X,y)
    pred = model.predict(X)
    models.append(model)
    # oof pred 
    for i,m in enumerate([0, 6, 12, 24]):
        train_s2[f'{target}_plus_{m}_s2'] = pred[len(train_s2)*i:len(train_s2)*(i+1)]
        #train_s2[f'{target}_plus_{m}_s2_t'] = y[len(train_s2)*i:len(train_s2)*(i+1)]
    print("for {} : SMAPE+1 : {}".format(target,smape_plus_1(y,pred)))

 not better for : updrs_3 & updrs_4

In [ ]:
for target in ["updrs_3","updrs_4"]:
    for i,m in enumerate([0, 6, 12, 24]):
        train_s2[f'{target}_plus_{m}_s2'] = train_s2[f'{target}_plus_{m}_s1']

In [ ]:
# eval oof pred
true_col = [f'{target}_plus_{plus_month}' for plus_month in [0, 6, 12, 24] for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']]
y_gt = train_s2[true_col].values.ravel()
pred_col = [f'{target}_plus_{plus_month}'+"_s2" for plus_month in [0, 6, 12, 24] for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']]
preds = train_s2[pred_col].values.ravel()
print("Stage2 oof pred : {}".format(smape_plus_1(y_gt,preds)))

# Prediction

In [ ]:
amp_pd_peptide.make_env.func_dict['__called__'] = False
env = amp_pd_peptide.make_env()   # initialize the environment
iter_test = env.iter_test()    # an iterator which loops over the test files

proteins_features_all = pd.DataFrame()
# The API will deliver four dataframes in this specific order:
for test_clinical_data, test_peptides, test_proteins, sample_submission in iter_test:
    sample_submission['patient_id'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[0]))
    sample_submission['visit_month'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[1]))
    sample_submission['target_name'] = sample_submission['prediction_id'].map(lambda x: 'updrs_' + x.split('_')[3])
    sample_submission['plus_month'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[5]))
    sample_submission['pred_month'] = sample_submission['visit_month'] + sample_submission['plus_month']
    sample_submission['visit_id'] = sample_submission['patient_id'].astype(str) + '_' + sample_submission['visit_month'].astype(str)
    
    # stage1
    for i in range(1, 5):
        target = f'updrs_{i}'
        mask_target = sample_submission['target_name'] == target
        sample_submission.loc[mask_target, 'target'] = calculate_predictions(
            pred_month=sample_submission.loc[mask_target, 'pred_month'],
            trend=target_to_trend[target],
            target=target
        )
    
    proteins_features = pd.pivot_table(test_proteins, values='NPX', index='visit_id', columns='UniProt', aggfunc='sum')
    proteins_features['visit_id'] = proteins_features.index
    proteins_features_all = pd.concat([proteins_features_all, proteins_features])
    proteins_features_all['patient_id'] = proteins_features_all.index.map(lambda x: int(x.split('_')[0]))
    proteins_features_all[proteins_features.columns] = proteins_features_all.groupby('patient_id')[proteins_features.columns].\
                                                                                                   fillna(method='ffill')
    proteins_features = proteins_features_all.groupby('patient_id', as_index=False).last()
    
    sample_submission = sample_submission.merge(
        proteins_features,
        on='patient_id',
        how='inner'
    )

    for i in range(1, 5):
        model = models[i-1]
        X = np.empty([len(train_s2)*4,len(features)+1])
        y = np.empty([len(train_s2)*4])
        target = f'updrs_{i}'
        for i,m in enumerate([0, 6, 12, 24]):
            X[len(train_s2)*i:len(train_s2)*(i+1),:-1] = train_s2[features].values
            X[len(train_s2)*i:len(train_s2)*(i+1),-1] = m
        pred = model.predict(X)
        # oof pred 
        for i,m in enumerate([0, 6, 12, 24]):
            train_s2[f'{target}_plus_{m}_s2'] = pred[len(train_s2)*i:len(train_s2)*(i+1)]
#         sample_submission.loc[mask_target, 'rating'] = np.round(sample_submission.loc[mask_target, 'rating'])
        
#     # call the env.predict for every iteration
    env.predict(sample_submission[['prediction_id', 'rating']])